<a href="https://colab.research.google.com/github/ariqRam/sotsuken/blob/main/hands_on_patching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "sbintuitions/sarashina2-7b"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto",
)

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.83MB            

tokenizer.model: downloading bytes:           |  0.00B            

pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [2]:
model, model.device

(LlamaForCausalLM(
   (model): LlamaModel(
     (embed_tokens): Embedding(102400, 4096)
     (layers): ModuleList(
       (0-31): 32 x LlamaDecoderLayer(
         (self_attn): LlamaAttention(
           (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
           (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
           (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
           (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
         )
         (mlp): LlamaMLP(
           (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
           (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
           (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
           (act_fn): SiLUActivation()
         )
         (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
         (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
       )
     )
     (norm): LlamaRMSNorm(

In [9]:
# -------------------------------------------------------------------------
# 1. Activation Capturing Hook
# -------------------------------------------------------------------------
captured_activations = {}

def get_capture_hook(layer_name: str):
    def hook(module, input_tensor, output_tensor):
        # Decoder layer outputs are usually a tuple: (hidden_states, ...)
        hidden_states = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
        # Detach and move to CPU to avoid VRAM overhead
        captured_activations[layer_name] = hidden_states.detach().cpu()
    return hook

# -------------------------------------------------------------------------
# 2. Activation Patching / Steering Hook
# -------------------------------------------------------------------------
# A patching hook modifies the tensor in-flight and returns the updated output.

def get_patching_hook(steering_vector: torch.Tensor, scale: float = 1.0, source_token_id=-1, base_token_id=-1):
    print("steering_vector shape:", steering_vector.shape)
    def hook(module, input_tensor, output_tensor):
        if isinstance(output_tensor, tuple):
            hidden_states = output_tensor[0]
            rest = output_tensor[1:]
        else:
            hidden_states = output_tensor
            rest = None

        # Apply intervention (e.g., add steering direction to the last token position)
        steered_states = hidden_states.clone()
        steered_states[:, base_token_id, :] = scale * steering_vector[:, source_token_id, :].to(hidden_states.device)

        if rest is not None:
            return (steered_states,) + rest
        return steered_states
    return hook

In [4]:
def get_next_token_probs(prompt):
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  with torch.no_grad():
      outputs = model(**inputs)

  # Extract logits for the next token position (last token of the sequence)
  next_token_logits = outputs.logits[0, -1, :]

  # Convert logits to probabilities
  probabilities = torch.softmax(next_token_logits, dim=-1)

  # Retrieve top 5 tokens and their probabilities
  top_k = 5
  top_probs, top_indices = torch.topk(probabilities, k=top_k)

  # Display ranked distribution
  print(f"Prompt: {prompt!r}\n")
  print(f"{'Rank':<5} | {'Token':<12} | {'Token ID':<10} | {'Probability':<12}")
  print("-" * 45)

  for rank, (prob, idx) in enumerate(zip(top_probs, top_indices), start=1):
      token_str = tokenizer.decode(idx)
      print(f"{rank:<5} | {repr(token_str):<12} | {idx.item():<10} | {prob.item():.4%}")

In [12]:
target_layer_idx = 1
hook_handle = model.model.layers[target_layer_idx].register_forward_hook(
    get_capture_hook(f"layer_{target_layer_idx}")
)

# Run forward pass (prompt inference)
prompt = "「犬」の鳴き声は「"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(inputs.token_to_word(0))

with torch.no_grad():
    outputs = model(**inputs)

print("Captured shape:", captured_activations[f"layer_{target_layer_idx}"].shape, captured_activations)
# Shape: [batch_size, sequence_length, hidden_dim]

# Clean up capture hook
hook_handle.remove()
hidden_dim = model.config.hidden_size
# steering_vector = torch.randn(hidden_dim, dtype=torch.float16) # altered by some random Gaussian noise
steering_vector = captured_activations[f"layer_{target_layer_idx}"]

patch_handle = model.model.layers[target_layer_idx].register_forward_hook(
    get_patching_hook(steering_vector, scale=1.0, source_token_id=1, base_token_id=1)
)

prompt = "「猫」の鳴き声は「"
try:
  get_next_token_probs(prompt)
except RuntimeError as e:
  print("RuntimeError", e)
  patch_handle.remove()

# print("\nGenerated with patch:")
# print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# # Clean up patching hook
patch_handle.remove()

0
Captured shape: torch.Size([1, 7, 4096]) defaultdict(<class 'dict'>, {'layer_1': {'mlp': tensor([[[ 0.1434, -0.3027,  0.2937,  ...,  0.0998, -0.1676,  0.0511],
         [-0.1707, -0.2246,  0.0836,  ...,  0.1456, -0.1088,  0.1901],
         [ 0.2878,  0.0444,  0.0834,  ...,  0.1136,  0.1980,  0.0811],
         ...,
         [ 0.0594,  0.0613, -0.0815,  ...,  0.2439, -0.0935,  0.0385],
         [ 0.0690, -0.0139, -0.0903,  ...,  0.1144, -0.2290,  0.0685],
         [ 0.6377, -0.0352,  0.1148,  ..., -0.0862, -0.3545,  0.0151]]],
       dtype=torch.float16), 'self_attn': tensor([[[ 0.1434, -0.3027,  0.2937,  ...,  0.0998, -0.1676,  0.0511],
         [-0.1707, -0.2246,  0.0836,  ...,  0.1456, -0.1088,  0.1901],
         [ 0.2878,  0.0444,  0.0834,  ...,  0.1136,  0.1980,  0.0811],
         ...,
         [ 0.0594,  0.0613, -0.0815,  ...,  0.2439, -0.0935,  0.0385],
         [ 0.0690, -0.0139, -0.0903,  ...,  0.1144, -0.2290,  0.0685],
         [ 0.6377, -0.0352,  0.1148,  ..., -0.0862, -0.3

In [ ]:
prompt = "「猫」の鳴き声は「"
tokenizer.tokenize(prompt)

In [ ]:
tokenizer.tokenize("「犬」の鳴き声は「")

In [ ]:
tokenizer.tokenize("猫が来た")

In [ ]:
tokenizer.tokenize("犬が来た")

In [ ]:
tokenizer.tokenize("犬の名前は")

In [ ]:
tokenizer.tokenize("猫の名前は")